In [1]:
import numpy as np

In [40]:

def pd_sim(x):

    # ===============================
    # Constants (km, s, kg)
    # ===============================
    N_DEF = 1
    SHIP_SPACING = 0              # km
    INITIAL_SEPARATION = 1000.0      # km
    DRIFT_SPEED = 1.0                # km/s

    # Railgun
    # SLUG_MASS = 512.0                # kg
    # SLUG_SPEED = 16.2                 # km/s
    # SHOTS_PER_SALVO = 5
    # INTRA_SALVO_CD = 7.2            # s
    # INTER_SALVO_CD = 36.0            # s

    SLUG_MASS = 875.0                # kg
    SLUG_SPEED = 6.                 # km/s
    SHOTS_PER_SALVO = 3
    INTRA_SALVO_CD = 18.            # s
    INTER_SALVO_CD = 24.0            # s

    N_ATTACKERS = 1

    # # Point defense
    # ENGAGE_RADIUS = 400             # km
    # PD_SHOTS_PER_SALVO = 8
    # PD_INTRA_CD = 2.0      # s
    # PD_INTER_CD = 8.0      # s
    # PD_EVAP = x                   # kg per hit                     # s
    # PD_PER_SHIP = 1

        # Point defense
    ENGAGE_RADIUS = 84             # km
    PD_SHOTS_PER_SALVO = 1
    PD_INTRA_CD = 0.0      # s
    PD_INTER_CD = 3.0      # s
    PD_EVAP = x                   # kg per hit                     # s
    PD_PER_SHIP = 5

    DT = 0.1                         # s

    # ===============================
    # Geometry
    # ===============================
    ship_offsets = (np.arange(N_DEF) - N_DEF // 2) * SHIP_SPACING
    pd_units = []
    for i, dx in enumerate(ship_offsets):
        max_range = np.sqrt(max(ENGAGE_RADIUS**2 - dx**2, 0.0))
        for _ in range(PD_PER_SHIP):
            pd_units.append({
                "ship_idx": i,
                "max_range": max_range,
                "next_fire": 0.0,
                "shots_left": PD_SHOTS_PER_SALVO
            })
    # print(pd_units)



    # ===============================
    # Slug fire schedule
    # ===============================
    salvo_times = []
    t_fire = 0.0
    while t_fire < 10_000:
        for k in range(SHOTS_PER_SALVO):
            salvo_times.append(t_fire + k * INTRA_SALVO_CD)
        t_fire += SHOTS_PER_SALVO * INTRA_SALVO_CD + INTER_SALVO_CD
    salvo_times = np.array(salvo_times)
    fire_ptr = 0

    # ===============================
    # State
    # ===============================
    t = 0.0
    slugs = []   # each: [x, mass]

    # ===============================
    # Simulation loop
    # ===============================
    hit = False
    t_clear = 0
    while not hit:
        # Relative drift
        separation = INITIAL_SEPARATION - DRIFT_SPEED * t

        # Fire railguns
        while fire_ptr < len(salvo_times) and t >= salvo_times[fire_ptr]:
            for _ in range(N_ATTACKERS):
                slugs.append([separation, SLUG_MASS])
            fire_ptr += 1

        # Move slugs
        for s in slugs:
            s[0] -= SLUG_SPEED * DT

        # Point defense
        for pd in pd_units:
            if t < pd["next_fire"]:
                continue

            for s in slugs:
                if s[1] <= 0:
                    continue

                if s[0] <= pd["max_range"]:
                    # Fire PD shot
                    s[1] -= PD_EVAP + 2*(np.random.rand()-0.5)*0.25*PD_EVAP
                    pd["shots_left"] -= 1

                    if pd["shots_left"] > 0:
                        pd["next_fire"] = t + PD_INTRA_CD
                    else:
                        pd["shots_left"] = PD_SHOTS_PER_SALVO
                        pd["next_fire"] = t + PD_INTER_CD
                    break
        # print(pd_units)


        # Remove destroyed slugs

        slugs = [s for s in slugs if s[1] > 0]
 

        # Check hit (slug crosses x=0)
        for s in slugs:
            if s[0] <= 0.0:
                hit = True
                # print("HIT")
                # print("time [s]:", t)
                # print("ship separations [km]:", separation)
                break

        t += DT

    return t, separation


In [42]:
mass = np.linspace(0, 100, num=101)

for m in mass:
    t,d = pd_sim(m)
    print(m,d)# time.append(t)

0.0 833.4000000000052
1.0 833.4000000000052
2.0 833.4000000000052
3.0 833.4000000000052
4.0 833.4000000000052
5.0 833.4000000000052
6.0 833.4000000000052
7.0 833.4000000000052
8.0 833.4000000000052
9.0 833.4000000000052
10.0 833.4000000000052
11.0 833.4000000000052
12.0 833.4000000000052
13.0 833.4000000000052
14.0 833.4000000000052
15.0 833.4000000000052
16.0 833.4000000000052
17.0 833.4000000000052
18.0 833.4000000000052
19.0 833.4000000000052
20.0 833.4000000000052
21.0 833.4000000000052
22.0 833.4000000000052
23.0 833.4000000000052
24.0 833.4000000000052
25.0 833.4000000000052
26.0 833.4000000000052
27.0 833.4000000000052
28.0 833.4000000000052
29.0 833.4000000000052
30.0 833.4000000000052
31.0 833.4000000000052
32.0 833.4000000000052
33.0 833.4000000000052
34.0 768.3000000000088
35.0 833.4000000000052
36.0 688.2999999999977
37.0 638.2999999999863
38.0 53.39999999985332
39.0 53.39999999985332
40.0 53.39999999985332
41.0 53.39999999985332
42.0 53.39999999985332
43.0 53.3999999998533

In [116]:
70*1.3/20

4.55